# IPNW — krzywe uczenia studentów ResNet-8

Porównanie supervised, bezpośredniej KD (110 → 8) i TAKD (110 → 32 → 8).
Wykres przedstawia **metryki walidacyjne**, bez wygładzania. Train loss nie jest porównywany, ponieważ supervised i KD mają różne funkcje celu.



In [ ]:
from pathlib import Path
import csv

# Działa przy uruchomieniu z katalogu głównego lub z notebooks/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'checkpoints').is_dir() and (path / 'pyproject.toml').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Uruchom notebook wewnątrz repozytorium teacher-assistant-selection.')

RUNS = {
    'Supervised': ('student-resnet8-baseline', '#475569'),
    'KD: 110 → 8': ('student-resnet8-kd', '#0072B2'),
    'TAKD: 110 → 32 → 8': ('student-resnet8-takd', '#D55E00'),
}
OUTPUT_DIR = PROJECT_ROOT / 'reports' / 'figures'
OUTPUT_PATH = OUTPUT_DIR / 'ipnw_krzywe_uczenia.png'


In [ ]:
# Wczytanie danych i sprawdzenie numeracji epok.
histories = {}
required = {'epoch', 'learning_rate', 'val_accuracy', 'val_loss'}

for label, (run_name, color) in RUNS.items():
    path = PROJECT_ROOT / 'checkpoints' / run_name / 'metrics.csv'
    with path.open(newline='', encoding='utf-8-sig') as file:
        reader = csv.DictReader(file)
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'{path}: brak kolumn {missing}')
        rows = [{key: float(value) for key, value in row.items()} for row in reader]
    if not rows:
        raise ValueError(f'{path}: pusty plik metryk')
    epochs = [int(row['epoch']) for row in rows]
    if any(row['epoch'] != int(row['epoch']) for row in rows):
        raise ValueError(f'{path}: niecałkowity numer epoki')
    if epochs != list(range(1, len(rows) + 1)):
        raise ValueError(f'{path}: brakujące, powtórzone lub nieuporządkowane epoki')
    histories[label] = rows
    best = max(rows, key=lambda row: row['val_accuracy'])
    print(f"{label}: {len(rows)} epok | najlepsze val accuracy: "
          f"{best['val_accuracy']:.2%} (epoka {int(best['epoch'])})")

# Zmiany LR są odczytywane z CSV, a nie wpisywane na sztywno.
lr_changes = {
    label: [int(current['epoch']) for previous, current in zip(rows, rows[1:])
            if current['learning_rate'] != previous['learning_rate']]
    for label, rows in histories.items()
}
print('Zmiany LR:', lr_changes)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

with plt.rc_context({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': '#94A3B8',
    'axes.labelcolor': '#475569',
    'xtick.color': '#475569',
    'ytick.color': '#475569',
}):
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    fig.subplots_adjust(top=0.74, bottom=0.18, left=0.07, right=0.97, wspace=0.22)
    fig.suptitle('CIFAR-100 | Porównanie studentów ResNet-8',
                 x=0.07, y=0.96, ha='left', fontsize=20, color='#18283B')
    fig.text(0.07, 0.89, 'Walidacja: 5 000 obrazów | seed 42 | bez wygładzania',
             color='#526174')

    for label, rows in histories.items():
        color = RUNS[label][1]
        epochs = [int(row['epoch']) for row in rows]
        axes[0].plot(epochs, [100 * row['val_accuracy'] for row in rows],
                     label=label, color=color, linewidth=1.5)
        axes[1].plot(epochs, [row['val_loss'] for row in rows],
                     label=label, color=color, linewidth=1.5)

    common_changes = len({tuple(values) for values in lr_changes.values()}) == 1
    for ax in axes:
        if common_changes:
            for epoch in next(iter(lr_changes.values())):
                ax.axvline(epoch, color='#9CA3AF', linestyle='--', linewidth=1, zorder=0)
        ax.grid(axis='y', color='#E2E8F0')
        ax.set_axisbelow(True)
        ax.set_xlabel('Epoka')
        ax.set_xlim(1, max(len(rows) for rows in histories.values()))
        ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))

    axes[0].set_title('Accuracy walidacyjna (%)', loc='left', pad=12)
    axes[0].set_ylim(0, max(70, max(100 * row['val_accuracy']
                                 for rows in histories.values() for row in rows) + 3))
    axes[1].set_title('Loss walidacyjny (cross-entropy)', loc='left', pad=12)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper left', bbox_to_anchor=(0.06, 0.85),
               ncol=3, frameon=False)
    note = 'Pojedynczy seed; wyniki nie dotyczą zbioru testowego.'
    if common_changes:
        changes = ', '.join(map(str, next(iter(lr_changes.values()))))
        note = f'Linie przerywane: zmiany LR od epok {changes}.\n' + note
    else:
        note = 'Przebiegi mają różne momenty zmian LR — patrz poprzednia komórka.\n' + note
    fig.text(0.07, 0.045, note, fontsize=10, color='#526174')

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_PATH, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'Zapisano: {OUTPUT_PATH}')
    plt.show()
    plt.close(fig)
